#### Calculation of FID

Only once this FID calculation code is written below. In other experiments FID is computed using modules.
$$
FID = || \mu_r - \mu_g ||^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})
$$


In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision.models import inception_v3
import torchvision.datasets as datasets
from torchvision import transforms
from scipy.linalg import sqrtm
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
# Generator needed to defined to load model
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            self.block(z_latent, fg * 8, 4, 1, 0),
            self.block(fg * 8, fg * 4, 4, 2, 1),
            self.block(fg * 4, fg * 2, 4, 2, 1),
            self.block(fg * 2, fg, 4, 2, 1),
            self.block(fg , fg//2, 4, 2, 1),
            nn.ConvTranspose2d(fg//2, n_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def block(self, in_channels, out_channels, kernal_size, stride, padding):
        return nn.Sequential(
            nn.ConvTranspose2d(
                in_channels, out_channels, kernal_size, stride, padding, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )
    def forward(self, z):
        return self.main(z)


In [ ]:
# Load model
G = torch.load('generator.pth', map_location=device )
G.eval()

# Load dataset
image_size = 128
path = 'data/augmented_animals'
mean = torch.tensor([0.50869316, 0.50057036, 0.44047505])
std = torch.tensor([0.20191325, 0.19737212, 0.20100889])
transform = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)
dataset = datasets.ImageFolder(root=path, transform=transform)

In [ ]:
# Generate 1000 fake images and save in fake_images list in tensor format.
latent_dim = 100
#dataset is loaded while calculating FID score
fake_images = []
for i in range(1000):
    noise = torch.randn(1, latent_dim, 1, 1).to(device)
    img = G(noise).detach().cpu().numpy().squeeze()
    # 0 to 1 normalization
    img = (img - img.min()) / (img.max() - img.min())
    fake_images.append(img)
fake_images = np.array(fake_images)

# 1000 real fake_images in tensor format
real_images = []
# sample 1000 images from the training set
for i in range(1000):
    img = dataset[i][0].numpy().squeeze()
    # 0 to 1 normalization
    img = (img - img.min()) / (img.max() - img.min())
    real_images.append(img)

In [14]:
# Define data paths
data = 'animals'
real_images_path = f'./real_{data}_1000'
fake_images_path = f'./fake_{data}_1000'

# Load pre-trained InceptionV3 model
model = inception_v3(pretrained=True, transform_input=False)
model.fc = nn.Identity()  # Remove the final classification layer
model = model.to(device)
model.eval()

# Define image transformation
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # Resize to 299x299 for InceptionV3
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.50869316, 0.50057036, 0.44047505], std=[0.20191325, 0.19737212, 0.20100889])
])

def get_activations(images, model, device):
    """Calculate activations using the InceptionV3 model."""
    activations = []

    for img in images:
        img = transform(img).unsqueeze(0).to(device)  # Apply transformation and add batch dimension
        with torch.no_grad():
            activation = model(img)
        activations.append(activation.cpu().numpy().squeeze())

    activations = np.array(activations)
    return activations

def calculate_fid_score(act1, act2):
    """Calculate the FID score between two sets of activations."""
    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)

    # Calculate sum squared difference between means
    ssdiff = np.sum((mu1 - mu2) ** 2.0)

    # Calculate sqrt of product between covariances
    covmean, _ = sqrtm(sigma1.dot(sigma2), disp=False)

    # Check and correct imaginary numbers from sqrtm
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    # Calculate FID score
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

def calculate_fid(real_images_path, fake_images_path):
    real_images_list = [Image.open(os.path.join(real_images_path, img)).convert('RGB') for img in os.listdir(real_images_path)]
    fake_images_list = [Image.open(os.path.join(fake_images_path, img)).convert('RGB') for img in os.listdir(fake_images_path)]

    real_activations = get_activations(real_images_list, model, device)
    fake_activations = get_activations(fake_images_list, model, device)

    fid_score = calculate_fid_score(real_activations, fake_activations)
    return fid_score

# Calculate and print the FID score
fid = calculate_fid(real_images_path, fake_images_path)
print(f'New GAN FID score: {fid}')

C:\Users\Shivam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Shivam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


New WGAN FID score: 199.5461323080913


Question 4 is over here.